# Advanced Python Descriptor Problems — Getters, Setters & Per-Instance State

**24 fully solved problems · runnable solutions · assertions · diagnostic examples · capstone**

**Source-based core:** This notebook expands the provided “Getters and Setters” lesson. That lesson demonstrates `__get__(self, instance, owner_class)`, class versus instance access, `@property`, descriptors shared by instances, the shared-state `Countdown` mistake, and `__set__(self, instance, value)` with a flawed `IntegerValue` that stores instance data on the descriptor. We preserve those ideas but use new, independently named implementations. **Advanced extensions** (not covered in the source lesson) include descriptor precedence, `__set_name__`, deletion, `__slots__`, weak references, inheritance, caching, validation, concurrency, SQLite, and transactional updates.

**Important precision:** `__set__` is normally triggered by assignment **through a managed instance** (`obj.x = value`). Ordinary assignment to the **class** (`Owner.x = value`) replaces the descriptor in that class's namespace rather than invoking its `__set__`.

**Workflow:** Run top to bottom in a fresh Python 3 kernel. For each numbered problem, first read the requirements, attempt it yourself, reveal the worked solution, execute it, and then run its test cell. Each problem uses a `Pxx` prefix to avoid accidental class-name collisions. No network access or third-party runtime libraries are required. Exercise code intentionally raising exceptions is wrapped in expectation checks so Run All completes successfully. Use a fresh kernel when repeating the full notebook.

**Conventions:** Instance-owned data goes in the instance `__dict__`, in a designated slot, or in a deliberately chosen weak-key mapping; configuration such as field name/range may live on the shared descriptor. For validation, use explicit error types, reject `bool` when an integer is intended, avoid silent coercion unless explicitly specified, and do not mutate stored values until validation succeeds.

## Learning path

| Track | Problems | Concepts |
|---|---:|---|
| A. Dispatch and state | 01–04 | `instance`, owner, UTC, shared-state failure, isolated storage |
| B. Lookup and lifecycle | 05–10 | data/non-data precedence, deletion, validation, inheritance, binding, slots |
| C. Subtle descriptor behavior | 11–16 | weak refs, recursion, bound methods, property, MRO, read-only fields |
| D. Production patterns | 17–20 | `Decimal`, auditing, lazy cache, cache invalidation |
| E. Integration and capstones | 21–24 | schema models, parameterized SQL, threads, atomic validation |

The **Quick recall**, **prediction drills**, and **final audit** reinforce the concepts. Each problem has a concrete specification, complete implementation, verification, explanatory solution, and optional extension.

In [1]:
from contextlib import contextmanager
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import gc
import inspect
import math
import sqlite3
import threading
import weakref
from concurrent.futures import ThreadPoolExecutor
from types import MethodType

@contextmanager
def expect_raises(expected):
    """Small stdlib-only negative-test helper; propagate unexpected errors."""
    try:
        yield
    except expected:
        pass
    else:
        raise AssertionError(f"Expected {expected.__name__} to be raised")

print("Setup complete: standard library only")

Setup complete: standard library only


## Track A — Dispatch and instance ownership

**Quick recall.** A descriptor is an object whose class defines `__get__`, `__set__`, or `__delete__`. On an ordinary instance lookup, Python's normal lookup algorithm gives **data descriptors** (those defining `__set__` or `__delete__`) precedence over the instance dictionary; **non-data descriptors** (only `__get__`) normally lose to values already in the instance dictionary. `instance is None` distinguishes class-level descriptor access from instance access for ordinary descriptor calls.

### Problem 01 — Trace the two access paths

Create a descriptor that records `(instance, owner)` for each `__get__`. Return the descriptor itself when accessed from a class; return `(type(instance).__name__, owner.__name__)` from an instance. Attach it to a base class, create a subclass, and account for what happens for `Base.field`, `Child.field`, and `Child().field`. Verify that `vars(Base)['field']` inspects the raw descriptor without running `__get__`.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [2]:
class P01Trace:
    def __init__(self):
        self.calls = []  # Diagnostic configuration/state for this one descriptor.

    def __get__(self, instance, owner):
        self.calls.append((instance, owner))
        if instance is None:
            return self
        return (type(instance).__name__, owner.__name__)

class P01Base:
    field = P01Trace()

class P01Child(P01Base):
    pass

p01_child = P01Child()
p01_descriptor = vars(P01Base)['field']
print('Raw descriptor:', type(p01_descriptor).__name__)

Raw descriptor: P01Trace


**Executable checks** — all assertions must pass.

In [3]:
assert P01Base.field is p01_descriptor
assert P01Child.field is p01_descriptor
assert p01_child.field == ('P01Child', 'P01Child')
assert p01_descriptor.calls[0] == (None, P01Base)
assert p01_descriptor.calls[1] == (None, P01Child)
assert p01_descriptor.calls[2] == (p01_child, P01Child)
assert vars(P01Base)['field'] is p01_descriptor
assert len(p01_descriptor.calls) == 3  # vars() did not invoke __get__.

print("Problem 01: PASS")

Problem 01: PASS


**Reasoning and best practices.** The *descriptor object* was created once in the base class. The `owner` passed by normal access is the class used in lookup (including a subclass), not necessarily the class in which the descriptor was first defined. Inspect `vars(cls)` to bypass the descriptor protocol for a directly defined attribute; for inherited raw lookup, use `inspect.getattr_static` later.

**Further challenge (optional).** Add a second base class and compare attribute lookup order via its MRO.

### Problem 02 — A testable, timezone-aware clock

Replace the lesson's `datetime.utcnow()` example with a timezone-aware, dependency-injected clock. Class access must return the descriptor; instance access must call the clock *on every read*, returning ISO 8601 text containing `+00:00`. No current-time-dependent assertion is permitted.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [4]:
class P02UTCClock:
    def __init__(self, clock):
        self.clock = clock

    def __get__(self, instance, owner):
        if instance is None:
            return self
        instant = self.clock()
        if instant.tzinfo is None or instant.utcoffset() is None:
            raise ValueError('Clock must return an aware datetime')
        return instant.astimezone(timezone.utc).isoformat()

p02_times = iter([
    datetime(2030, 1, 2, 3, 4, tzinfo=timezone.utc),
    datetime(2030, 1, 2, 3, 5, tzinfo=timezone.utc),
])

class P02Logger:
    current_time = P02UTCClock(lambda: next(p02_times))

**Executable checks** — all assertions must pass.

In [5]:
assert isinstance(P02Logger.current_time, P02UTCClock)
p02_logger = P02Logger()
assert p02_logger.current_time == '2030-01-02T03:04:00+00:00'
assert p02_logger.current_time == '2030-01-02T03:05:00+00:00'
class P02BadLogger:
    current_time = P02UTCClock(lambda: datetime(2030, 1, 1))
with expect_raises(ValueError):
    _ = P02BadLogger().current_time

print("Problem 02: PASS")

Problem 02: PASS


**Reasoning and best practices.** Clock injection makes time-dependent behavior deterministic. Aware UTC is preferable to naive `utcnow()` strings when consumers need to distinguish a UTC instant from local time. The time series is deliberately consumed only on instance access; checking the descriptor through the class does not advance it.

### Problem 03 — Diagnose AND repair the shared-countdown bug

The lesson's `Countdown` keeps its decreasing counter on the class-level descriptor, so two rockets share one countdown. First reproduce that failure. Then build `P03IndependentCountdown` with separate state per rocket. The first read for *each* rocket must return 3, followed by 2 and 1. Do not use a mutable descriptor field as rocket state.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [6]:
class P03BrokenCountdown:
    def __init__(self, start):
        self.remaining = start + 1  # WRONG: the descriptor is shared.

    def __get__(self, instance, owner):
        if instance is None:
            return self
        self.remaining -= 1
        return self.remaining

class P03BrokenRocket:
    countdown = P03BrokenCountdown(3)

class P03IndependentCountdown:
    def __init__(self, start):
        self.start = start  # Shared immutable configuration is fine.

    def __set_name__(self, owner, name):
        self.storage_name = f'_{name}_remaining'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        state = instance.__dict__
        current = state.get(self.storage_name, self.start + 1) - 1
        state[self.storage_name] = current
        return current

class P03Rocket:
    countdown = P03IndependentCountdown(3)

**Executable checks** — all assertions must pass.

In [7]:
p03_bad_a, p03_bad_b = P03BrokenRocket(), P03BrokenRocket()
assert (p03_bad_a.countdown, p03_bad_b.countdown) == (3, 2)
p03_a, p03_b = P03Rocket(), P03Rocket()
assert P03Rocket.countdown is vars(P03Rocket)['countdown']
assert (p03_a.countdown, p03_b.countdown) == (3, 3)
assert (p03_a.countdown, p03_b.countdown) == (2, 2)
assert p03_a.__dict__ == {'_countdown_remaining': 2}
assert p03_b.__dict__ == {'_countdown_remaining': 2}

print("Problem 03: PASS")

Problem 03: PASS


**Reasoning and best practices.** The class owns exactly one descriptor, but each rocket owns a separate dictionary entry. A read has a **side effect** in this exercise, so a production API might prefer `rocket.tick()` over a surprising property; this problem deliberately mirrors the lesson to make descriptor state ownership visible.

**Further challenge (optional).** Make the countdown stop at zero instead of returning negative values.

### Problem 04 — Strict reusable integer fields

Implement a `P04Integer` data descriptor usable twice on a 2-D point. Unset reads raise `AttributeError`; assignment accepts genuine integers, rejects `True` and floats rather than silently truncating them, and must isolate both fields AND both point instances. Use `__set_name__` so the same class works for x and y.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [8]:
class P04Integer:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError(f'{self.public_name} has not been set') from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError(f'{self.public_name} requires an int (not bool)')
        instance.__dict__[self.storage_name] = value

class P04Point:
    x = P04Integer()
    y = P04Integer()

    def __init__(self, x, y):
        self.x = x
        self.y = y

**Executable checks** — all assertions must pass.

In [9]:
p04_a, p04_b = P04Point(1, 2), P04Point(3, 4)
assert (p04_a.x, p04_a.y, p04_b.x, p04_b.y) == (1, 2, 3, 4)
p04_b.x = 99
assert (p04_a.x, p04_b.x) == (1, 99)
assert P04Point.x.public_name == 'x'
assert P04Point.y.public_name == 'y'
with expect_raises(TypeError):
    p04_a.x = 1.9
with expect_raises(TypeError):
    p04_a.y = True
with expect_raises(AttributeError):
    _ = P04Point.__new__(P04Point).x
assert (p04_a.x, p04_a.y) == (1, 2)

print("Problem 04: PASS")

Problem 04: PASS


**Reasoning and best practices.** Unlike the lesson's deliberately flawed implementation, this stores the value in the **point**, not `self._value` on the descriptor. The check uses `type(value) is int`: `bool` inherits from `int`, so `isinstance(True, int)` would inadvertently accept a boolean. Validating before writing makes failed assignments non-destructive.

## Track B — Descriptor precedence and lifecycle

The next problems use explicit dictionary injection only as a *diagnostic*, not as normal application code. It demonstrates lookup precedence and the fact that directly mutating `obj.__dict__` bypasses many managed-attribute checks.

### Problem 05 — Predict precedence: non-data versus data descriptor

Create two descriptors returning a marker on reads. `P05NonData` has only `__get__`; `P05Data` also has a raising `__set__`. Insert conflicting keys directly into the instance dictionary and predict each lookup. Separately show that ordinary `Class.attr = ...` replaces a class descriptor; it does NOT invoke that descriptor's `__set__`.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [10]:
class P05NonData:
    def __get__(self, instance, owner):
        return self if instance is None else 'non-data descriptor'

class P05Data:
    def __get__(self, instance, owner):
        return self if instance is None else 'data descriptor'

    def __set__(self, instance, value):
        raise AttributeError('read-only')

class P05Example:
    lazy = P05NonData()
    locked = P05Data()

p05_obj = P05Example()
p05_obj.__dict__.update(lazy='instance value', locked='instance value')

**Executable checks** — all assertions must pass.

In [11]:
assert p05_obj.lazy == 'instance value'
assert p05_obj.locked == 'data descriptor'
assert isinstance(P05Example.lazy, P05NonData)
assert isinstance(P05Example.locked, P05Data)
with expect_raises(AttributeError):
    p05_obj.locked = 'cannot assign'
class P05Replacement:
    field = P05Data()
p05_raw = vars(P05Replacement)['field']
P05Replacement.field = 'replaced at class level'
assert P05Replacement.field == 'replaced at class level'
assert p05_raw is not vars(P05Replacement)['field']

print("Problem 05: PASS")

Problem 05: PASS


**Reasoning and best practices.** A data descriptor outranks an instance dictionary entry even if its setter refuses changes. A non-data descriptor can be shadowed by the instance dictionary. Normal metaclass behavior handles assignment to the class namespace; descriptor `__set__` manages instance assignment. If class-level assignment itself must be validated, that is a *different*, metaclass-level design problem.

### Problem 06 — Implement deletion and well-defined unset state

Make a data descriptor implementing `__get__`, `__set__`, and `__delete__`. Reading or deleting an unset field raises `AttributeError`; deleting an assigned field removes only that object's value; repeated delete fails. Tests must leave a separate instance unchanged.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [12]:
class P06Deletable:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.key not in instance.__dict__:
            raise AttributeError(f'{self.name} is unset')
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value

    def __delete__(self, instance):
        if self.key not in instance.__dict__:
            raise AttributeError(f'{self.name} is already unset')
        del instance.__dict__[self.key]

class P06Record:
    value = P06Deletable()

**Executable checks** — all assertions must pass.

In [13]:
p06_a, p06_b = P06Record(), P06Record()
with expect_raises(AttributeError):
    _ = p06_a.value
p06_a.value, p06_b.value = 10, 20
assert (p06_a.value, p06_b.value) == (10, 20)
del p06_a.value
with expect_raises(AttributeError):
    _ = p06_a.value
with expect_raises(AttributeError):
    del p06_a.value
assert p06_b.value == 20

print("Problem 06: PASS")

Problem 06: PASS


**Reasoning and best practices.** `del obj.value` dispatches to a data descriptor's `__delete__`. Missing-value behavior should be explicit: raising `AttributeError` matches normal Python attribute semantics and makes `hasattr` behave naturally. Delete only the requested instance's backing entry.

### Problem 07 — Bounded validation with strong failure behavior

Build `P07BoundedInt(low, high)`, validating its constructor range, excluding bool, and rejecting out-of-range assignments. A rejected assignment must leave the previous value unchanged. Use two fields configured with different ranges; test lower and upper inclusive boundaries and both bad configuration and bad data.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [14]:
class P07BoundedInt:
    def __init__(self, low, high):
        if type(low) is not int or type(high) is not int or low > high:
            raise ValueError('bounds must be integers with low <= high')
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.key = f'_{name}'
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f'{self.name} is unset') from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError(f'{self.name} must be an int')
        if not self.low <= value <= self.high:
            raise ValueError(f'{self.name} must be within [{self.low}, {self.high}]')
        instance.__dict__[self.key] = value

class P07Game:
    lives = P07BoundedInt(0, 9)
    level = P07BoundedInt(1, 100)

**Executable checks** — all assertions must pass.

In [15]:
p07 = P07Game()
p07.lives, p07.level = 0, 100
assert (p07.lives, p07.level) == (0, 100)
with expect_raises(ValueError):
    p07.lives = 10
with expect_raises(ValueError):
    p07.level = 0
with expect_raises(TypeError):
    p07.lives = False
assert (p07.lives, p07.level) == (0, 100)
assert (P07Game.lives.low, P07Game.level.high) == (0, 100)
with expect_raises(ValueError):
    P07BoundedInt(5, 4)

print("Problem 07: PASS")

Problem 07: PASS


**Reasoning and best practices.** The descriptor safely shares immutable bounds but not mutable per-game values. Each `__set__` validates everything before updating the instance, so failures preserve earlier state. The constructor validates descriptor *configuration* separately from its assignment logic.

### Problem 08 — Understand `__set_name__` and inherited owner

Record the owner/name passed to `__set_name__` when defining a base class. Then subclass it without redeclaring the descriptor. Show that the descriptor's recorded defining owner remains the base, even though `__get__` receives the subclass on ordinary subclass access.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [16]:
class P08Binding:
    def __init__(self):
        self.binding_events = []
        self.access_events = []

    def __set_name__(self, owner, name):
        self.binding_events.append((owner, name))

    def __get__(self, instance, owner):
        self.access_events.append((instance, owner))
        return self if instance is None else owner.__name__

class P08Base:
    field = P08Binding()

class P08Child(P08Base):
    pass

p08_descriptor = vars(P08Base)['field']

**Executable checks** — all assertions must pass.

In [17]:
assert p08_descriptor.binding_events == [(P08Base, 'field')]
assert P08Child.field is p08_descriptor
assert P08Child().field == 'P08Child'
assert p08_descriptor.binding_events == [(P08Base, 'field')]
assert p08_descriptor.access_events == [(None, P08Child), (p08_descriptor.access_events[1][0], P08Child)]
assert isinstance(p08_descriptor.access_events[1][0], P08Child)

print("Problem 08: PASS")

Problem 08: PASS


**Reasoning and best practices.** `__set_name__` is called during creation of a class containing the descriptor in *its own* namespace. Inheritance of that descriptor does not call it again. By contrast, the `owner` argument to normal `__get__` reflects the accessing subclass. Avoid assuming these two owner meanings are interchangeable.

### Problem 09 — Reject accidental reuse of one descriptor instance

A descriptor that stores one `storage_name` on itself cannot safely be assigned to `first` and `second` under different names. Implement a binding guard that rejects *explicit rebinding* of the very same descriptor instance, including across classes. It should allow normal inheritance. Account for Python wrapping exceptions raised in `__set_name__` during class creation.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [18]:
class P09SingleBinding:
    def __init__(self):
        self.bound_to = None

    def __set_name__(self, owner, name):
        if self.bound_to is not None:
            raise TypeError(f'Already bound to {self.bound_to[1]}')
        self.bound_to = (owner, name)
        self.key = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.key)

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value

p09_shared = P09SingleBinding()
class P09Good:
    first = p09_shared
class P09Inherited(P09Good):
    pass

**Executable checks** — all assertions must pass.

In [19]:
assert P09Inherited.first is p09_shared
assert p09_shared.bound_to == (P09Good, 'first')
try:
    class P09Invalid:
        a = p09_shared
except (RuntimeError, TypeError) as exc:
    original = exc.__cause__ or exc
    assert isinstance(original, TypeError)
    assert 'Already bound' in str(original)
else:
    raise AssertionError('Explicit rebinding should fail')
p09_new = P09SingleBinding()
try:
    class P09AlsoInvalid:
        left = p09_new
        right = p09_new
except (RuntimeError, TypeError) as exc:
    assert isinstance(exc.__cause__ or exc, TypeError)
else:
    raise AssertionError('Two names for the same descriptor should fail')

print("Problem 09: PASS")

Problem 09: PASS


**Reasoning and best practices.** When the same instance is reused, a naïve `__set_name__` overwrites its key; assignments can then silently route to the wrong field. Rejecting reuse is one design choice; another is maintaining a carefully designed `(owner, name)` mapping. Class creation can expose the original exception or wrap it in `RuntimeError` with the original as `__cause__`, depending on Python/runtime version; the checks handle both.

### Problem 10 — Support classes using `__slots__`

A slotted object might not have `__dict__`. Store validated x/y through explicitly declared private slots rather than a mapping on the descriptor. Test that a second object is independent, assignment works, and direct `__dict__` access fails. Make missing reads raise `AttributeError`.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [20]:
class P10SlotInteger:
    def __set_name__(self, owner, name):
        self.slot = f'_{name}'
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return getattr(instance, self.slot)
        except AttributeError:
            raise AttributeError(f'{self.name} has not been assigned') from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError('int required')
        setattr(instance, self.slot, value)

class P10SlottedPoint:
    __slots__ = ('_x', '_y')  # x and y themselves are descriptors.
    x = P10SlotInteger()
    y = P10SlotInteger()

    def __init__(self, x, y):
        self.x, self.y = x, y

**Executable checks** — all assertions must pass.

In [21]:
p10_a, p10_b = P10SlottedPoint(1, 2), P10SlottedPoint(3, 4)
assert (p10_a.x, p10_a.y, p10_b.x, p10_b.y) == (1, 2, 3, 4)
p10_b.x = 8
assert (p10_a.x, p10_b.x) == (1, 8)
with expect_raises(AttributeError):
    _ = p10_a.__dict__
with expect_raises(TypeError):
    p10_a.x = 1.5
with expect_raises(AttributeError):
    _ = P10SlottedPoint.__new__(P10SlottedPoint).x

print("Problem 10: PASS")

Problem 10: PASS


**Reasoning and best practices.** Slots change the storage model: a descriptor that blindly uses `instance.__dict__` fails for dictionary-free objects. These private slots are instance-owned and their names do not collide with the public descriptors. Because `getattr(instance, '_x')` targets a different name, it does not recurse into `x`.

## Track C — Weak references, binding, and method lookup

These cases explore alternative storage, common recursion failures, ordinary function binding, and the relation to `property`. Some choices have explicit caveats (especially weak-key maps).

### Problem 11 — Weak-key storage without retaining objects

Build a writable descriptor backed by `weakref.WeakKeyDictionary`. Show that different instances have independent values and that an instance may be garbage-collected even after being assigned a value. State the constraints: instances must support weak references and be hashable; equality semantics can affect weak-key maps.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [22]:
class P11WeakField:
    def __init__(self):
        self.values = weakref.WeakKeyDictionary()

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return self.values[instance]
        except KeyError:
            raise AttributeError('value is unset') from None

    def __set__(self, instance, value):
        self.values[instance] = value

    def __delete__(self, instance):
        try:
            del self.values[instance]
        except KeyError:
            raise AttributeError('value is unset') from None

class P11Person:
    score = P11WeakField()

**Executable checks** — all assertions must pass.

In [23]:
p11_a, p11_b = P11Person(), P11Person()
p11_a.score, p11_b.score = 12, 34
assert (p11_a.score, p11_b.score) == (12, 34)
p11_map = P11Person.score.values
assert len(p11_map) == 2
p11_reference = weakref.ref(p11_a)
del p11_a
gc.collect()
assert p11_reference() is None
assert len(p11_map) == 1
assert p11_b.score == 34
class P11NoWeakRef:
    __slots__ = ()
    value = P11WeakField()
with expect_raises(TypeError):
    P11NoWeakRef().value = 1

print("Problem 11: PASS")

Problem 11: PASS


**Reasoning and best practices.** A weak-key map can support instances whose storage is not in `__dict__`, but it adds constraints and overhead. A weak key does **not** prevent collection only when the mapped value does not itself strongly reference its key (directly or indirectly). A class with `__slots__` must generally include `__weakref__`, and default object identity/hash behavior is the least surprising keying policy here.

**Further challenge (optional).** Try a dataclass with `eq=True` and no hash to observe why some instances cannot be weak-map keys.

### Problem 12 — Find and fix recursive descriptor lookup

Explain why a descriptor that tries `getattr(instance, 'value')` inside its own getter recurses. Confirm the failure with an expected-exception test, then implement the correct backing-name lookup. Do not catch a recursion error inside a production descriptor as a storage strategy.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [24]:
class P12Recursive:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, 'value')  # Calls this same __get__ again.

class P12Broken:
    value = P12Recursive()

class P12Safe:
    def __set_name__(self, owner, name):
        self.storage = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        instance.__dict__[self.storage] = value

class P12Fixed:
    value = P12Safe()

**Executable checks** — all assertions must pass.

In [25]:
with expect_raises(RecursionError):
    _ = P12Broken().value
p12 = P12Fixed()
p12.value = 42
assert p12.value == 42
assert p12.__dict__ == {'_value': 42}
assert P12Fixed.value is vars(P12Fixed)['value']

print("Problem 12: PASS")

Problem 12: PASS


**Reasoning and best practices.** Descriptor getters should retrieve the *backing* storage, not their own public managed name. Recursion is a symptom of confusing public access with raw storage. A dictionary lookup is appropriate for dictionary-based instances; slot-based designs use a distinct declared slot.

### Problem 13 — Reconstruct bound-method behavior

Functions are non-data descriptors. Make a descriptor that wraps a stored function with `types.MethodType` on instance access but yields itself on class access. Verify the resulting bound method receives the instance as `self` and does not retain a descriptor-created per-instance counter.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [26]:
class P13Method:
    def __init__(self, func):
        self.func = func

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self.func, instance)

def p13_greet(self, punctuation='!'):
    return f'Hello {self.name}{punctuation}'

class P13Greeter:
    greet = P13Method(p13_greet)

    def __init__(self, name):
        self.name = name

**Executable checks** — all assertions must pass.

In [27]:
p13_a, p13_b = P13Greeter('Ada'), P13Greeter('Grace')
assert P13Greeter.greet is vars(P13Greeter)['greet']
assert p13_a.greet() == 'Hello Ada!'
assert p13_b.greet('?') == 'Hello Grace?'
assert p13_a.greet.__self__ is p13_a
assert p13_b.greet.__func__ is p13_greet
assert p13_a.greet is not p13_a.greet  # Fresh bound-method objects may be created.

print("Problem 13: PASS")

Problem 13: PASS


**Reasoning and best practices.** A bound method is a function plus a bound receiver. This is why `obj.method(...)` transparently passes the instance, while accessing an ordinary function on the class yields the function. Returning the descriptor on class access here is an educational design choice for our wrapper; native functions instead return the function itself on class access.

### Problem 14 — Compare `property` to a custom data descriptor

Implement a property with a getter and setter using a private attribute. Compare class access, instance access, and shadowing via the dictionary to a read-only property. Explain why even a property with no setter counts as a data descriptor.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [28]:
class P14Temperature:
    def __init__(self, celsius):
        self._celsius = float(celsius)

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        numeric = float(value)
        if not math.isfinite(numeric):
            raise ValueError('finite temperature required')
        self._celsius = numeric

    @property
    def kelvin(self):
        return self._celsius + 273.15

**Executable checks** — all assertions must pass.

In [29]:
p14 = P14Temperature(20)
assert isinstance(P14Temperature.celsius, property)
assert isinstance(P14Temperature.kelvin, property)
assert p14.celsius == 20.0
p14.celsius = 25
assert p14.kelvin == 298.15
p14.__dict__['kelvin'] = -999
assert p14.kelvin == 298.15  # property takes precedence.
with expect_raises(AttributeError):
    p14.kelvin = -1
with expect_raises(ValueError):
    p14.celsius = float('nan')
assert p14.celsius == 25.0

print("Problem 14: PASS")

Problem 14: PASS


**Reasoning and best practices.** `property` implements the descriptor protocol and follows the same instance-versus-class access model used in the source lesson. Read-only properties still define a setter pathway that raises `AttributeError`, so they are *data descriptors* and win over conflicting instance-dictionary entries. Conversions must not accidentally accept non-finite values.

### Problem 15 — Inheritance, shadowing, and raw introspection

Put a data descriptor in a base class. Create one subclass that inherits it and a second that overrides the same name with a plain class constant. Show which descriptor is found under each lookup and how `inspect.getattr_static` returns raw attributes without calling `__get__`.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [30]:
class P15Label:
    def __get__(self, instance, owner):
        return self if instance is None else f'managed by {owner.__name__}'

    def __set__(self, instance, value):
        raise AttributeError('managed and read-only')

class P15Base:
    label = P15Label()

class P15Inherited(P15Base):
    pass

class P15Overridden(P15Base):
    label = 'plain subclass constant'

**Executable checks** — all assertions must pass.

In [31]:
p15 = P15Inherited()
p15.__dict__['label'] = 'injected'
assert p15.label == 'managed by P15Inherited'
assert P15Inherited.label is vars(P15Base)['label']
assert inspect.getattr_static(P15Inherited, 'label') is vars(P15Base)['label']
assert P15Overridden().label == 'plain subclass constant'
assert inspect.getattr_static(P15Overridden, 'label') == 'plain subclass constant'
with expect_raises(AttributeError):
    p15.label = 'not allowed'

print("Problem 15: PASS")

Problem 15: PASS


**Reasoning and best practices.** Normal MRO search chooses the subclass's own name before the base's attribute; an inherited descriptor only runs if lookup finds it. When debugging, `inspect.getattr_static` helps inspect the raw attribute without triggering either a descriptor's side effects or a custom `__getattribute__`.

### Problem 16 — Read-only instance values and the class-rebinding caveat

Implement a constant-like data descriptor that allows reads through an instance but rejects assignment and deletion there. Then demonstrate in a **separate class** that replacing a class attribute bypasses `__set__`. Explain why this makes a descriptor alone insufficient to freeze an entire class namespace.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [32]:
class P16ReadOnly:
    def __init__(self, value):
        self.value = value

    def __get__(self, instance, owner):
        return self if instance is None else self.value

    def __set__(self, instance, value):
        raise AttributeError('instance field is read-only')

    def __delete__(self, instance):
        raise AttributeError('instance field cannot be deleted')

class P16Settings:
    api_version = P16ReadOnly('v2')

class P16Replaceable:
    version = P16ReadOnly('original')

**Executable checks** — all assertions must pass.

In [33]:
p16 = P16Settings()
assert p16.api_version == 'v2'
with expect_raises(AttributeError):
    p16.api_version = 'v3'
with expect_raises(AttributeError):
    del p16.api_version
p16.__dict__['api_version'] = 'injected'
assert p16.api_version == 'v2'
assert isinstance(P16Settings.api_version, P16ReadOnly)
P16Replaceable.version = 'class was changed'
assert P16Replaceable().version == 'class was changed'

print("Problem 16: PASS")

Problem 16: PASS


**Reasoning and best practices.** The descriptor protects accesses mediated by its presence in the class lookup path; it cannot stop arbitrary mutation of its owning class namespace. Protecting class-level rebinding usually requires a carefully designed metaclass or a different API. The restriction is on the managed **instance** attribute, not on Python reflection or all possible forms of mutation.

## Track D — Validation, observability, and caching

Good descriptor APIs specify conversion rules, error types, and what happens after failed assignments. Some caching schemes intentionally use non-data descriptors so the cached instance value shadows the descriptor. Others deliberately remain data descriptors and keep cache state behind a private name.

### Problem 17 — Money validation with exact decimal arithmetic

Create a reusable monetary descriptor. Accept `Decimal`, `int` (but not bool), and decimal strings; reject binary floats, invalid strings, NaN, Infinity, and negatives. Round to two decimal places using `ROUND_HALF_UP`, and leave the old value unchanged on any rejection. Test two products for isolated state.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [34]:
class P17Money:
    CENT = Decimal('0.01')

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f'{self.name} has not been set') from None

    def __set__(self, instance, value):
        if type(value) is int:
            amount = Decimal(value)
        elif isinstance(value, (str, Decimal)):
            try:
                amount = Decimal(value)
            except InvalidOperation as exc:
                raise ValueError('invalid decimal input') from exc
        else:
            raise TypeError('use int, str, or Decimal; floats are disallowed')
        if not amount.is_finite() or amount < 0:
            raise ValueError('amount must be finite and nonnegative')
        try:
            normalized = amount.quantize(self.CENT, rounding=ROUND_HALF_UP)
        except InvalidOperation as exc:
            raise ValueError('amount cannot be represented at cent precision') from exc
        instance.__dict__[self.key] = normalized

class P17Product:
    price = P17Money()

    def __init__(self, price):
        self.price = price

**Executable checks** — all assertions must pass.

In [35]:
p17_a, p17_b = P17Product('1.005'), P17Product(2)
assert p17_a.price == Decimal('1.01')
assert p17_b.price == Decimal('2.00')
p17_b.price = Decimal('0.004')
assert p17_b.price == Decimal('0.00')
for invalid, error in [
    (1.25, TypeError), (True, TypeError), ('nope', ValueError),
    ('NaN', ValueError), ('Infinity', ValueError), ('-0.01', ValueError),
]:
    with expect_raises(error):
        p17_a.price = invalid
    assert p17_a.price == Decimal('1.01')
assert p17_b.price == Decimal('0.00')
assert isinstance(P17Product.price, P17Money)

print("Problem 17: PASS")

Problem 17: PASS


**Reasoning and best practices.** Binary floats may already contain an approximation before they reach a descriptor, so this API rejects them instead of silently converting. All parsing, finite-value checks, nonnegativity checks, and quantization complete before the instance dictionary is changed. Monetary rules such as maximum amount, currencies, and rounding mode depend on the application and should be stated explicitly.

**Further challenge (optional).** Add a separate, validated currency code and prohibit arithmetic across currencies.

### Problem 18 — Audit successful changes without shared history

Implement an integer field whose successful assignments record `(old, new)` tuples per instance. Unset old values should be `None`. Failed assignments must change neither the stored value nor the audit history. Return an immutable snapshot of history, not the internal list itself.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [36]:
class P18AuditedInt:
    def __set_name__(self, owner, name):
        self.name = name
        self.value_key = f'_{name}'
        self.history_key = f'_{name}_history'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.value_key not in instance.__dict__:
            raise AttributeError(f'{self.name} is unset')
        return instance.__dict__[self.value_key]

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError('int required')
        state = instance.__dict__
        previous = state.get(self.value_key)
        state[self.value_key] = value
        state.setdefault(self.history_key, []).append((previous, value))

    def history(self, instance):
        return tuple(instance.__dict__.get(self.history_key, ()))

class P18Account:
    points = P18AuditedInt()

**Executable checks** — all assertions must pass.

In [37]:
p18_a, p18_b = P18Account(), P18Account()
p18_a.points = 10
p18_a.points = 15
p18_b.points = 7
assert P18Account.points.history(p18_a) == ((None, 10), (10, 15))
assert P18Account.points.history(p18_b) == ((None, 7),)
with expect_raises(TypeError):
    p18_a.points = 2.1
assert p18_a.points == 15
assert P18Account.points.history(p18_a) == ((None, 10), (10, 15))
snapshot = P18Account.points.history(p18_a)
with expect_raises(AttributeError):
    snapshot.append((15, 20))

print("Problem 18: PASS")

Problem 18: PASS


**Reasoning and best practices.** The descriptor stores only field-name configuration; the value and audit list belong to each account. Validation precedes mutation. An immutable tuple snapshot prevents callers from directly rewriting the history list. This is a simple in-memory audit, **not** a crash-safe transaction or security log; concurrent writers would need synchronization, and durable systems need persistent storage.

### Problem 19 — A non-data lazy cached descriptor

Implement a lazy descriptor that computes an expensive property once per instance and caches it under the **public name** in that instance's dictionary. Show that the cached value shadows a non-data descriptor, that instances compute separately, and that deleting the dictionary entry invalidates the cache.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [38]:
class P19Cached:
    def __init__(self, func):
        self.func = func

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = self.func(instance)
        instance.__dict__[self.name] = value
        return value

class P19Report:
    def __init__(self, seed):
        self.seed = seed
        self.compute_count = 0

    @P19Cached
    def total(self):
        self.compute_count += 1
        return self.seed * 10

**Executable checks** — all assertions must pass.

In [39]:
p19_a, p19_b = P19Report(3), P19Report(4)
assert isinstance(P19Report.total, P19Cached)
assert p19_a.total == 30
assert p19_a.total == 30
assert p19_b.total == 40
assert (p19_a.compute_count, p19_b.compute_count) == (1, 1)
assert p19_a.__dict__['total'] == 30
p19_a.seed = 5
assert p19_a.total == 30  # Cached values can become stale.
del p19_a.__dict__['total']
assert p19_a.total == 50
assert p19_a.compute_count == 2
assert p19_b.compute_count == 1

print("Problem 19: PASS")

Problem 19: PASS


**Reasoning and best practices.** This intentionally has only `__get__`, making it a non-data descriptor: after the first read, the instance dictionary wins over the descriptor. Such caching is simple and fast but needs an invalidation policy when inputs change. It is not inherently thread-safe: two concurrent first reads can compute more than once.

**Further challenge (optional).** Add a TTL or explicit `invalidate(instance)` method and document its time/concurrency semantics.

### Problem 20 — Data-descriptor caching with correct invalidation

Implement a rectangle with validated positive finite `width` and `height` fields. Its read-only `area` descriptor caches a result in `_area_cache`, not under `area`. Changing either dimension must invalidate the cache **only after** successful validation. Test an invalid update, an attempted direct assignment to area, and independence of two rectangles.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [40]:
class P20PositiveDimension:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f'{self.name} is unset') from None

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError('a real numeric dimension is required')
        number = float(value)
        if not math.isfinite(number) or number <= 0:
            raise ValueError('dimension must be finite and positive')
        instance.__dict__[self.key] = number
        instance.__dict__.pop('_area_cache', None)

class P20Area:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        if '_area_cache' not in instance.__dict__:
            instance._area_computations += 1
            instance.__dict__['_area_cache'] = instance.width * instance.height
        return instance.__dict__['_area_cache']

    def __set__(self, instance, value):
        raise AttributeError('area is derived and read-only')

class P20Rectangle:
    width = P20PositiveDimension()
    height = P20PositiveDimension()
    area = P20Area()

    def __init__(self, width, height):
        self._area_computations = 0
        self.width = width
        self.height = height

**Executable checks** — all assertions must pass.

In [41]:
p20_a, p20_b = P20Rectangle(2, 3), P20Rectangle(4, 5)
assert p20_a.area == 6.0
assert p20_a.area == 6.0
assert p20_a._area_computations == 1
assert p20_b.area == 20.0
assert p20_b._area_computations == 1
p20_a.width = 7
assert p20_a.area == 21.0
assert p20_a._area_computations == 2
with expect_raises(ValueError):
    p20_a.width = -1
assert p20_a.width == 7.0 and p20_a.area == 21.0
assert p20_a._area_computations == 2
with expect_raises(AttributeError):
    p20_a.area = 999
p20_a.__dict__['area'] = 999  # Injection cannot shadow a data descriptor.
assert p20_a.area == 21.0
assert p20_b.area == 20.0

print("Problem 20: PASS")

Problem 20: PASS


**Reasoning and best practices.** The width/height setters validate before updating the dimension and invalidating the area cache. Area defines `__set__` that raises, so an injected `area` dictionary entry cannot shadow it. This separates the public derived attribute from private cache storage. The example provides single-threaded consistency; simultaneous writes would require stronger synchronization.

## Track E — Integration and capstones

These problems compose descriptor concepts with a model schema, database APIs, and thread coordination. Only the following integrations use external *systems* conceptually; all required Python modules remain in the standard library. SQL identifiers are fixed programmatically and SQL **values** are always bound as parameters.

### Problem 21 — Collect validated inherited fields into a schema

Build a small `Model` framework with a metaclass that collects descriptor fields in definition order and inherits fields from parent models. Each field validates on assignment; model initialization rejects missing or unknown arguments; `to_dict()` returns validated values. Make a base user model and an employee subclass. Avoid invoking descriptors accidentally while building the class registry.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [42]:
class P21Field:
    def __init__(self, validator):
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f'{self.name} has not been set') from None

    def __set__(self, instance, value):
        validated = self.validator(value)
        instance.__dict__[self.key] = validated

def p21_name(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError('name must be nonempty text')
    return value.strip()

def p21_positive_int(value):
    if type(value) is not int or value <= 0:
        raise ValueError('must be a positive int')
    return value

class P21ModelMeta(type):
    def __new__(mcls, name, bases, namespace):
        fields = {}
        for base in bases:
            fields.update(getattr(base, '__fields__', {}))
        fields.update({key: value for key, value in namespace.items()
                       if isinstance(value, P21Field)})
        cls = super().__new__(mcls, name, bases, namespace)
        cls.__fields__ = fields
        return cls

class P21Model(metaclass=P21ModelMeta):
    def __init__(self, **values):
        required = set(type(self).__fields__)
        supplied = set(values)
        if required != supplied:
            raise TypeError(f'missing={sorted(required - supplied)}, '
                            f'unknown={sorted(supplied - required)}')
        for name, value in values.items():
            setattr(self, name, value)

    def to_dict(self):
        return {name: getattr(self, name) for name in type(self).__fields__}

class P21User(P21Model):
    name = P21Field(p21_name)

class P21Employee(P21User):
    employee_id = P21Field(p21_positive_int)

**Executable checks** — all assertions must pass.

In [43]:
assert list(P21User.__fields__) == ['name']
assert list(P21Employee.__fields__) == ['name', 'employee_id']
p21_employee = P21Employee(name='  Ada  ', employee_id=7)
assert p21_employee.to_dict() == {'name': 'Ada', 'employee_id': 7}
assert isinstance(P21Employee.name, P21Field)
with expect_raises(TypeError):
    P21Employee(name='Grace')
with expect_raises(TypeError):
    P21Employee(name='Grace', employee_id=3, secret='unknown')
with expect_raises(ValueError):
    P21Employee(name='', employee_id=3)
with expect_raises(ValueError):
    p21_employee.employee_id = 0
assert p21_employee.employee_id == 7

print("Problem 21: PASS")

Problem 21: PASS


**Reasoning and best practices.** The metaclass reads the **class namespace** and explicit inherited `__fields__` registries rather than accessors on instances. `__set_name__` supplies each public field's backing key. Constructor validation checks the complete argument set before assignment; however, a failure while assigning *later fields* can leave a partially initialized object, so this small model is not a fully transactional constructor. The final capstone addresses all-or-nothing updates.

**Further challenge (optional).** Support overriding a field in a child with a plain non-field attribute by removing that name from the inherited schema.

### Problem 22 — Descriptor-validated data and parameterized SQLite

Create a descriptor-validated `Event` with a nonempty title and positive capacity, then persist two events in an in-memory SQLite table. One title contains SQL-looking text: prove that it remains ordinary data and does not delete the table. Use parameter placeholders for all user-supplied values; never interpolate a title into SQL.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [44]:
class P22Field:
    def __init__(self, validator):
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        normalized = self.validator(value)
        instance.__dict__[self.key] = normalized

def p22_title(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError('title cannot be empty')
    return value.strip()

def p22_capacity(value):
    if type(value) is not int or value <= 0:
        raise ValueError('capacity must be a positive int')
    return value

class P22Event:
    title = P22Field(p22_title)
    capacity = P22Field(p22_capacity)

    def __init__(self, title, capacity):
        self.title = title
        self.capacity = capacity

p22_events = [
    P22Event('Python workshop', 30),
    P22Event("A'); DROP TABLE events; --", 12),
]

p22_connection = sqlite3.connect(':memory:')
try:
    p22_connection.execute(
        'CREATE TABLE events (id INTEGER PRIMARY KEY, title TEXT NOT NULL, '
        'capacity INTEGER NOT NULL)'
    )
    p22_connection.executemany(
        'INSERT INTO events (title, capacity) VALUES (?, ?)',
        [(event.title, event.capacity) for event in p22_events],
    )
    p22_rows = p22_connection.execute(
        'SELECT title, capacity FROM events ORDER BY id'
    ).fetchall()
    p22_table_count = p22_connection.execute(
        "SELECT count(*) FROM sqlite_master WHERE type='table' AND name='events'"
    ).fetchone()[0]
finally:
    p22_connection.close()

**Executable checks** — all assertions must pass.

In [45]:
assert p22_rows == [
    ('Python workshop', 30),
    ("A'); DROP TABLE events; --", 12),
]
assert p22_table_count == 1
assert p22_events[0].title == 'Python workshop'
with expect_raises(ValueError):
    p22_events[0].capacity = -1
assert p22_events[0].capacity == 30

print("Problem 22: PASS")

Problem 22: PASS


**Reasoning and best practices.** Descriptors validate and normalize the model's Python-side state. SQL placeholders (`?`) separately protect the database boundary by binding *values* rather than parsing them as SQL. Validation and parameterization solve different problems, so do both. Table and column identifiers are fixed trusted strings in this example; dynamically supplied identifiers need their own careful allowlist.

### Problem 23 — A per-instance counter safe under competing readers

The source countdown is stateful; extend that pattern for concurrent reads. Create a descriptor that returns incrementing values `1, 2, ...` using a **per-instance** lock and counter. Read it concurrently with eight worker threads and verify the results contain each integer exactly once. Explain what guarantee the lock provides and what it does not.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [46]:
class P23AtomicCounter:
    def __set_name__(self, owner, name):
        self.key = f'_{name}_current'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        with instance._counter_lock:
            state = instance.__dict__
            next_value = state.get(self.key, 0) + 1
            state[self.key] = next_value
            return next_value

class P23Worker:
    counter = P23AtomicCounter()

    def __init__(self):
        self._counter_lock = threading.Lock()

p23_a, p23_b = P23Worker(), P23Worker()
with ThreadPoolExecutor(max_workers=8) as executor:
    p23_results_a = list(executor.map(lambda unused: p23_a.counter, range(400)))
    p23_results_b = list(executor.map(lambda unused: p23_b.counter, range(300)))

**Executable checks** — all assertions must pass.

In [47]:
assert sorted(p23_results_a) == list(range(1, 401))
assert sorted(p23_results_b) == list(range(1, 301))
assert p23_a.__dict__['_counter_current'] == 400
assert p23_b.__dict__['_counter_current'] == 300
assert p23_a.counter == 401
assert p23_b.counter == 301
assert isinstance(P23Worker.counter, P23AtomicCounter)

print("Problem 23: PASS")

Problem 23: PASS


**Reasoning and best practices.** Each worker owns its own lock and count. The lock makes the read/modify/write sequence indivisible **relative to other users of that same lock** and prevents duplicate increments even when execution interleaves. This is intentionally a *per-instance* lock, not a descriptor-wide global bottleneck. It does not make arbitrary direct edits to `__dict__` safe, nor does it confer consistency across multiple unrelated fields or processes.

### Problem 24 — Capstone: defaults, defensive copies, and all-or-nothing updates

Create reusable typed fields that support a validated `default_factory`. Make a settings object with positive finite timeout, nonnegative integer retries (bool rejected), and a list of nonempty tag strings. Defaults must never share a mutable tag list. Implement `update(**changes)` so it (1) rejects unknown names, (2) validates **all** input before committing **any** change, (3) normalizes/copies lists to prevent caller aliasing, and (4) keeps old values unchanged if one field is invalid. Test success, rollback, and isolation.

**Task:** Predict the behavior and/or implement the required descriptor before reading the solution. Write down which object holds each mutable value.

**Worked solution** — run the next cell, then inspect the assertions.

In [48]:
class P24Field:
    def __init__(self, normalize, default_factory=None):
        self.normalize = normalize
        self.default_factory = default_factory

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f'_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.key not in instance.__dict__:
            if self.default_factory is None:
                raise AttributeError(f'{self.name} is required')
            default = self.normalize(self.default_factory())
            instance.__dict__[self.key] = default
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        validated = self.normalize(value)
        instance.__dict__[self.key] = validated

def p24_timeout(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError('timeout must be numeric')
    number = float(value)
    if not math.isfinite(number) or number <= 0:
        raise ValueError('timeout must be positive and finite')
    return number

def p24_retries(value):
    if type(value) is not int:
        raise TypeError('retries must be an int, not bool')
    if value < 0:
        raise ValueError('retries must be nonnegative')
    return value

def p24_tags(value):
    if not isinstance(value, list):
        raise TypeError('tags must be a list')
    if any(not isinstance(tag, str) or not tag.strip() for tag in value):
        raise ValueError('tags must contain only nonempty strings')
    return [tag.strip() for tag in value]  # Defensive copy.

class P24Settings:
    timeout = P24Field(p24_timeout, default_factory=lambda: 1.0)
    retries = P24Field(p24_retries, default_factory=lambda: 3)
    tags = P24Field(p24_tags, default_factory=list)

    def update(self, **changes):
        descriptors = {
            name: field for name, field in vars(type(self)).items()
            if isinstance(field, P24Field)
        }
        unknown = set(changes) - set(descriptors)
        if unknown:
            raise KeyError(f'unknown fields: {sorted(unknown)}')

        staged = {}  # No live instance mutation during validation.
        for name, candidate in changes.items():
            field = descriptors[name]
            staged[field.key] = field.normalize(candidate)

        self.__dict__.update(staged)  # Commit after all checks succeeded.

p24_a, p24_b = P24Settings(), P24Settings()

**Executable checks** — all assertions must pass.

In [49]:
assert p24_a.timeout == 1.0 and p24_a.retries == 3
assert p24_a.tags == [] and p24_b.tags == []
assert p24_a.tags is not p24_b.tags
caller_tags = ['  api ', 'core']
p24_a.update(timeout=2, retries=0, tags=caller_tags)
assert (p24_a.timeout, p24_a.retries, p24_a.tags) == (2.0, 0, ['api', 'core'])
caller_tags.append('outside mutation')
assert p24_a.tags == ['api', 'core']
prior = (p24_a.timeout, p24_a.retries, list(p24_a.tags))
with expect_raises(ValueError):
    p24_a.update(timeout=9, retries=-1, tags=['looks valid'])
assert (p24_a.timeout, p24_a.retries, p24_a.tags) == prior
with expect_raises(KeyError):
    p24_a.update(timeout=5, arbitrary='unknown')
assert (p24_a.timeout, p24_a.retries, p24_a.tags) == prior
with expect_raises(TypeError):
    p24_a.update(retries=True)
assert p24_a.retries == 0
assert (p24_b.timeout, p24_b.retries, p24_b.tags) == (1.0, 3, [])
assert isinstance(P24Settings.tags, P24Field)

print("Problem 24: PASS")

Problem 24: PASS


**Reasoning and best practices.** The descriptor enforces a reusable boundary for *individual assignments* and generates one default per instance. The multi-field update stages all normalized values without invoking setters or mutating the object, then commits them together with a dictionary update. This gives **validation-level all-or-nothing behavior in the intended single-threaded model**, not a database transaction or cross-thread atomic snapshot. A caller can still mutate the returned `tags` list; to enforce deep immutability, expose a tuple or a defensive copy on reads.

**Further challenge (optional).** Make tags externally immutable (`tuple`) and add an explicit append API; optionally add a per-instance lock around reads/updates for a stronger concurrency contract.

---
## Eight extra prediction drills (with answers)

For each one, predict the value or error **before** running the code. Each drill is independent. These short examples supplement the 24 larger implementations.

| Drill | Question to decide |
|---|---|
| A | What does a class-access getter receive as `instance`? |
| B | Does writing a class attribute call a descriptor setter? |
| C | Can a non-data descriptor be shadowed? |
| D | Can a data descriptor be shadowed by `__dict__`? |
| E | Does inheritance invoke `__set_name__` again? |
| F | Can `hasattr` report `False` for an unset field? |
| G | Is `bool` an `int` subclass? |
| H | Does class-level access to a custom time descriptor consume the clock? |

In [50]:
# Drill A — class access supplies instance=None (through normal class lookup).
class DrillA:
    class D:
        def __get__(self, instance, owner):
            return (instance, owner)
    field = D()
assert DrillA.field == (None, DrillA)

# Drill B — class rebinding replaces the class's descriptor in ordinary classes.
class DrillB:
    field = P05Data()
DrillB.field = 'replacement'
assert DrillB.field == 'replacement'

# Drill C — non-data descriptor loses to the instance dictionary.
class DrillC:
    field = P05NonData()
c = DrillC()
c.field = 'instance'
assert c.field == 'instance'

# Drill D — data descriptor wins, including over manual dictionary injection.
class DrillD:
    field = P05Data()
d = DrillD()
d.__dict__['field'] = 'injected'
assert d.field == 'data descriptor'

# Drill E — only the defining namespace triggers __set_name__ here.
class DrillEBase:
    field = P08Binding()
class DrillEChild(DrillEBase):
    pass
assert vars(DrillEBase)['field'].binding_events == [(DrillEBase, 'field')]

# Drill F — AttributeError means hasattr() returns False.
assert not hasattr(P06Record(), 'value')

# Drill G — bool is an int subclass; use strict checks when needed.
assert isinstance(True, int) and type(True) is not int

# Drill H — P02Logger's descriptor is accessible without evaluating its clock.
assert isinstance(P02Logger.current_time, P02UTCClock)
print('All 8 prediction drills: PASS')

All 8 prediction drills: PASS


### Prediction-drill answer key

**A:** `(None, DrillA)`; class access communicates the absence of an instance. **B:** a plain class attribute replaces the original class descriptor. **C:** the instance value wins against a non-data descriptor. **D:** the data descriptor wins. **E:** inheriting a descriptor does not repeat its `__set_name__` notification. **F:** yes, because `hasattr` interprets `AttributeError` as missing. **G:** yes, `bool` subclasses `int`, so use a strict type test if required. **H:** no; the design returns `self` for class access, so it does not call the clock.

## Final engineering checklist

- [ ] State is stored per managed instance, not accidentally on a shared descriptor.
- [ ] Class access behavior is intentional and consistent (`instance is None`).
- [ ] The owner argument and `__set_name__` defining owner are not confused.
- [ ] Data versus non-data precedence is understood, including class reassignment.
- [ ] Invalid input is rejected *before* mutating live values or invalidating caches.
- [ ] Missing reads, deletion, and read-only behavior raise appropriate exceptions.
- [ ] Type policy (including `bool`), ranges, normalization, and finite numeric values is explicit.
- [ ] `__dict__`, slots, and weak keys are chosen based on instance capabilities and lifecycle.
- [ ] Caches specify invalidation and concurrency behavior.
- [ ] External persistence parameterizes user values; validation is not a substitute.
- [ ] Multi-field operations state their actual atomicity and locking guarantees.

**Self-assessment:** Explain aloud why the original lesson's shared `Countdown` and descriptor-held `_value` fail for multiple objects, then compare at least three storage choices from the solutions (`__dict__`, slots, and weak keys). If you can justify those tradeoffs and pass all checks in Run All, you've covered the notebook's core and advanced learning objectives.

In [51]:
# Final executable smoke check: check the key end-to-end invariants.
assert (p03_a.countdown, p03_b.countdown) == (1, 1)
assert (p04_a.x, p04_b.x) == (1, 99)
assert P18Account.points.history(p18_a) == ((None, 10), (10, 15))
assert p20_a.area == 21.0 and p20_b.area == 20.0
assert p21_employee.to_dict() == {'name': 'Ada', 'employee_id': 7}
assert p22_table_count == 1
assert p24_a.tags == ['api', 'core'] and p24_b.tags == []
print('FINAL STATUS: all solved problems and supplemental checks passed.')

FINAL STATUS: all solved problems and supplemental checks passed.
